# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore the FAIR^2 dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

We will reference all schema entities (record sets, fields/columns, etc.) by their `@id` according to FAIR and Croissant best practices.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata. This step fetches schema, structure, and main descriptive information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print("Description:")
print(metadata.description)
print("\nDataset identifier:", getattr(metadata, 'identifier', ''))
print("Version:", getattr(metadata, 'version', ''))
print("License:", getattr(metadata, 'license', ''))

## 2. Data Overview
Review available **record sets** and their **fields/columns**.

We examine all `recordSet` entries in the schema, printing their `@id`, and for each, the associated fields' and columns' `@id` and name. This helps determine which sets are available for extraction.

**Note:** You may inspect the list of record sets to decide which to load.

In [ ]:
# Explore Record Sets and their Fields by @id
def describe_record_sets(metadata):
    print("Available record sets:")
    for record_set in getattr(metadata, 'recordSet', []):
        rid = getattr(record_set, '@id', None)
        print(f"Record set: {rid} | name: {getattr(record_set, 'name', '')}")
        fields = getattr(record_set, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            fid = getattr(field, '@id', None)
            label = getattr(field, 'name', getattr(field, 'description', ''))
            dtype = getattr(field, 'dataType', '')
            print(f"   Field @id: {fid} | name: {label} | type: {dtype}")
        columns = getattr(record_set, 'column', [])
        if not isinstance(columns, list):
            columns = [columns]
        for col in columns:
            colid = getattr(col, '@id', None)
            col_name = getattr(col, 'name', getattr(col, 'description', ''))
            print(f"   Column @id: {colid} | name: {col_name}")
        print()
    if not getattr(metadata, 'recordSet', []):
        print("No explicit top-level recordSet found in schema.")

describe_record_sets(metadata)

## 3. Data Extraction
Select a record set by `@id` and extract its records to a pandas DataFrame.

- List available record set IDs above, pick one (replace `record_set_id`).
- All data extraction refers to `@id`, never field index or position.

In [ ]:
# Find available record set IDs for extraction
record_sets = [getattr(rs, '@id', None) for rs in getattr(metadata, 'recordSet', [])]
record_sets = [rsid for rsid in record_sets if rsid]

if not record_sets:
    print("No record sets available for extraction in Croissant schema. Please inspect the schema JSON directly if needed.")
else:
    print("Available record_set @id(s):")
    pprint.pprint(record_sets)

# --- Example record set selection (replace below with discovered @id, e.g. 'cr:recordSet_1') ---
# If the @id is unknown, you may need to load the records with the main dataset (some datasets put all table rows as main records)
# For demonstration, we fallback to the first available record set if present, otherwise try main level extraction

dataframes = {}

if record_sets:
    selected_record_set = record_sets[0]
    print(f"\nExtracting records from record set: {selected_record_set}")
    records = list(dataset.records(record_set=selected_record_set))
    df = pd.DataFrame(records)
    dataframes[selected_record_set] = df
else:
    # Try loading records from the root level if no record set is found
    print("Attempting to extract records from main dataset...")
    try:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['main'] = df
        selected_record_set = 'main'
    except Exception as e:
        print("Could not extract records:", e)
        df = pd.DataFrame()
        selected_record_set = None

if not df.empty:
    print(f"Columns in DataFrame ({selected_record_set}):")
    print(df.columns.tolist())
    display_cols = df.columns[:10]
    print(df[display_cols].head())
else:
    print("No data records extracted.")

## 4. Exploratory Data Analysis (EDA)
Let's post-process extracted records:
- Pick a **numeric field by its `@id` or column name** you saw above for analysis.
- Example EDA: filter, normalize, and group by another key field (e.g., protein group, experiment).

Adjust the column names to refer to those present in your DataFrame (from the output above).

In [ ]:
# ------------- User: replace these variables after inspecting DataFrame columns above -------------
# If a column 'MW [kDa]' (molecular weight), or 'percent_coverage', etc. exist, use their column name 

# Example: suppose column 'Molecular_Weight_kDa' exists, or pick any numeric field from df.columns
numeric_field = None
group_field = None

# Try auto-detect a numeric field
for col in df.columns:
    if str(df[col].dtype).startswith('float') or str(df[col].dtype).startswith('int'):
        numeric_field = col
        break
# Try to find a group field
for col in df.columns:
    if (df[col].dtype == 'object') and (col != numeric_field):
        group_field = col
        break

if numeric_field is None:
    print("No numeric column found for EDA; please inspect DataFrame and manually set `numeric_field`.")
else:
    print(f"Using numeric_field='{numeric_field}'")

    # Set threshold for demonstration (e.g., mean value)
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (N={len(filtered_df)} records):")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[numeric_field + '_normalized'] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    if group_field is not None and group_field in df.columns:
        # Group and aggregate (mean)
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No suitable group field detected for aggregation. Set `group_field` manually if needed.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the group field.

You may customize the plot for your analysis needs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is None or df.empty:
    print("No numeric field or data for visualization.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
    
    if group_field is not None and group_field in df.columns:
        # Show top N groups by count
        top_groups = df[group_field].value_counts().head(5).index.tolist()
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (Top 5 groups)")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, you have seen how to:
- Load FAIR^2 dataset metadata and records using `mlcroissant`
- Enumerate and select record sets and fields via their `@id`
- Extract, inspect, and process the data in pandas
- Run simple exploratory data analysis and visualizations

Refer to the [mlcroissant documentation](https://mlcroissant.readthedocs.io/en/latest/) for advanced use: joining fields, provenance, harmonization, large file handling, and more.